In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
# Define the directory containing the CSV files
csv_dir = '../../results/loo_patient'

# List all CSV files in the directory
csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]

# Read and concatenate all CSV files into a single DataFrame
df_list = []
for csv_file in csv_files:
    df = pd.read_csv(os.path.join(csv_dir, csv_file))
    df_list.append(df)
combined_df = pd.concat(df_list, ignore_index=True)


In [3]:
combined_df['rmse_lfc'] = np.sqrt(combined_df['mse_lfc'])
combined_df = combined_df.rename(columns={"direction_match_k": "signed_precision"})
combined_df = combined_df.rename(columns={"edistance_pca_log": "e-distance"})

In [4]:
metrics = ['pearson', 'signed_precision', 'e-distance', 'rmse_lfc']

In [5]:
summary = combined_df.groupby("model_name")[metrics].agg(["mean", "std"])

# Format as "mean ± std" strings
table = pd.DataFrame(index=summary.index)
for m in metrics:
    table[m] = summary[(m, "mean")].round(2).astype(str) + " ± " + summary[(m, "std")].round(2).astype(str)

print(table)

                pearson signed_precision     e-distance     rmse_lfc
model_name                                                          
baseline    0.45 ± 0.24      0.15 ± 0.16  30.98 ± 10.34  4.85 ± 2.55
cellina     0.79 ± 0.13        0.3 ± 0.2    6.55 ± 1.18  1.37 ± 0.68
cpa         0.52 ± 0.18       0.12 ± 0.2   15.05 ± 4.22  2.88 ± 0.76


## Statistical significance testing

Cellina vs. cpa and cellina vs. baseline, paired by (slide, held-out celltype) — 30 pairs per comparison.
One-sided paired Wilcoxon signed-rank test per metric (direction chosen so the alternative is "cellina is better"), with Holm-Bonferroni correction across all 8 tests (2 comparisons × 4 metrics).

In [6]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# Whether a higher value of the metric means a better prediction
higher_is_better = {
    "pearson": True,
    "signed_precision": True,
    "e-distance": False,
    "rmse_lfc": False,
}

def paired_values(df, model_a, model_b, metric):
    """Align per-(sid, holdout_celltype) scores for two models on one metric."""
    a = df[df.model_name == model_a].set_index(["sid", "holdout_celltype"])[metric]
    b = df[df.model_name == model_b].set_index(["sid", "holdout_celltype"])[metric]
    common = a.index.intersection(b.index)
    return a.loc[common], b.loc[common]

def win_rate(a, b, higher_better):
    wins = (a.values > b.values) if higher_better else (a.values < b.values)
    return wins.sum() / len(a)

comparisons = [("cellina", "cpa"), ("cellina", "baseline")]

rows = []
for model_a, model_b in comparisons:
    for metric in metrics:
        a, b = paired_values(combined_df, model_a, model_b, metric)
        alternative = "greater" if higher_is_better[metric] else "less"
        stat, p = wilcoxon(a, b, alternative=alternative)
        rows.append({
            "model_a": model_a,
            "model_b": model_b,
            "metric": metric,
            "n_pairs": len(a),
            "mean_a": a.mean(),
            "mean_b": b.mean(),
            f"{model_a}_win_rate": win_rate(a, b, higher_is_better[metric]),
            "statistic": stat,
            "p_value": p,
        })

stats_df = pd.DataFrame(rows)

# Holm-Bonferroni correction across the full family of tests (2 comparisons x 4 metrics)
reject, p_adj, _, _ = multipletests(stats_df["p_value"], method="holm")
stats_df["p_adj"] = p_adj
stats_df["significant"] = reject

stats_df

,model_a,model_b,metric,n_pairs,mean_a,mean_b,cellina_win_rate,statistic,p_value,p_adj,significant
0,cellina,cpa,pearson,30,0.789130,0.519610,0.933333,456.0,3.073364e-08,1.229346e-07,True
1,cellina,cpa,signed_precision,30,0.298667,0.124667,0.900000,432.0,1.748283e-06,3.496566e-06,True
2,cellina,cpa,e-distance,30,6.545477,15.052658,1.000000,0.0,9.313226e-10,7.450581e-09,True
3,cellina,cpa,rmse_lfc,30,1.370548,2.879698,0.966667,3.0,4.656613e-09,2.328306e-08,True
4,cellina,baseline,pearson,30,0.789130,0.452769,0.966667,455.0,4.004687e-08,1.229346e-07,True
5,cellina,baseline,signed_precision,30,0.298667,0.150000,0.833333,422.0,4.809459e-06,4.809459e-06,True
6,cellina,baseline,e-distance,30,6.545477,30.981452,1.000000,0.0,9.313226e-10,7.450581e-09,True
7,cellina,baseline,rmse_lfc,30,1.370548,4.852940,1.000000,0.0,9.313226e-10,7.450581e-09,True


In [7]:
display_df = stats_df.copy()
display_df["comparison"] = display_df["model_a"] + " vs " + display_df["model_b"]
display_df["p_value"] = display_df["p_value"].map(lambda p: f"{p:.2e}")
display_df["p_adj"] = display_df["p_adj"].map(lambda p: f"{p:.2e}")
display_df = display_df.set_index(["comparison", "metric"])[
    ["n_pairs", "mean_a", "mean_b", "statistic", "p_value", "p_adj", "significant"]
]
display_df

n_pairs    mean_a     mean_b  statistic  \
comparison          metric                                                      
cellina vs cpa      pearson                30  0.789130   0.519610      456.0   
                    signed_precision       30  0.298667   0.124667      432.0   
                    e-distance             30  6.545477  15.052658        0.0   
                    rmse_lfc               30  1.370548   2.879698        3.0   
cellina vs baseline pearson                30  0.789130   0.452769      455.0   
                    signed_precision       30  0.298667   0.150000      422.0   
                    e-distance             30  6.545477  30.981452        0.0   
                    rmse_lfc               30  1.370548   4.852940        0.0   

                                       p_value     p_adj  significant  
comparison          metric                                             
cellina vs cpa      pearson           3.07e-08  1.23e-07         True  
                    signed_precision  1.75e-06  3.50e-06         True  
                    e-distance        9.31e-10  7.45e-09         True  
                    rmse_lfc          4.66e-09  2.33e-08         True  
cellina vs baseline pearson           4.00e-08  1.23e-07         True  
                    signed_precision  4.81e-06  4.81e-06         True  
                    e-distance        9.31e-10  7.45e-09         True  
                    rmse_lfc          9.31e-10  7.45e-09         True